### Imports

In [1]:
import os
import cv2
import torch
import numpy as np
from utils import *
import pandas as pd
import seaborn as sns
from modules import *
from PIL import Image
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
import utils.for_eye_tracker as et
from scipy.signal import savgol_filter
from torchvision import models, transforms
from transformers import CLIPProcessor, CLIPModel
from torchvision.models.segmentation import DeepLabV3_ResNet101_Weights
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

Current city is copenhagen!
If you wish to change the city, please edit the value in the __init__.py file


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/heartpy/datautils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [2]:
data_root = "/mnt/raid/emotional_data_raquel/data"
fix_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations"

_________

## Gaze Preprocessing and Fixation Extraction (IVT)

In [3]:
def process_gaze_to_fixations(csv_path):

    # --- LOAD ---
    df = pd.read_csv(csv_path)

    df = df.rename(columns={
        'Seconds': 'time',
        'GazeX': 'x',
        'GazeY': 'y'
    })

    df['time'] = pd.to_datetime(df['time'])

    df = df[['time', 'x', 'y']]

    # --- CLEAN (drop duplicates and have positive x and y values) ---
    # df = df.drop_duplicates(subset='time')
    # if same time → drop and keep same, if different time → average x&y values
    df = (df.groupby('time', as_index=False)[['x', 'y']].mean())
    df = df[(df['x'] >= 0) & (df['y'] >= 0)]

    # --- PREPROCESS ---
    df = df.sort_values('time')

    # only fill gaps (NaNs) of up to 5 consecutive missing samples
    df['x'] = df['x'].interpolate(limit=5)
    df['y'] = df['y'].interpolate(limit=5)

    # drop long gaps (too much of a stretch)
    df = df.dropna(subset=['x', 'y'])

    if len(df) > 5:
        # savgol smooths noise without destroying real movement patterns
        df['x_smooth'] = savgol_filter(df['x'], 5, 2)
        df['y_smooth'] = savgol_filter(df['y'], 5, 2)
    else:
        # if too little data → don’t smooth, just keep raw
        df['x_smooth'] = df['x']
        df['y_smooth'] = df['y']

    # --- VELOCITY ---
    # convert timestamps to seconds for velocity calculation (dt)
    t_sec = df['time'].astype('int64') / 1e9

    # differences between consecutive samples
    dt = np.diff(t_sec) # how much time passed
    dx = np.diff(df['x_smooth']) # how much gaze moved
    dy = np.diff(df['y_smooth']) # how much gaze moved

    # if two timestamps are identical, ignore this step (shouldnt happen but)
    dt[dt == 0] = np.nan

    # velocity = sqrt(dx² + dy²) / dt
    velocity = np.sqrt(dx**2 + dy**2) / dt

    df['velocity'] = np.nan
    # assign velocity to rows 1..N (each value corresponds to movement from previous row)
    df.iloc[1:, df.columns.get_loc('velocity')] = velocity

    # --- IVT CLASSIFICATION ---
    df['event'] = 'fixation'
    # high velocity = saccade, otherwise fixation
    df.loc[df['velocity'] > 1000, 'event'] = 'saccade'

    # --- EXTRACT FIXATIONS ---
    fixations = []
    current_fix = None

    for idx in range(len(df)):
        row = df.iloc[idx]
        if row['event'] == 'fixation':
            # storing when a fixation start time is, if before wasnt a fixation
            if current_fix is None:
                current_fix = {
                    'start_time': row['time'],
                    'x': [],
                    'y': []
                }

            # continuing a fixation, if before as already a fixation
            current_fix['x'].append(row['x'])
            current_fix['y'].append(row['y'])

        else:
            # only act if we were in a fixation and now is a saccade, 
            # and calculate end time, durantion & means
            if current_fix is not None:
                end_time = df.iloc[idx-1]['time']

                fixations.append({
                    'start_time': current_fix['start_time'],
                    'end_time': end_time,
                    'duration': (end_time - current_fix['start_time']).total_seconds(),
                    'x_mean': np.mean(current_fix['x']),
                    'y_mean': np.mean(current_fix['y'])
                })

                current_fix = None

    # --- HANDLE LAST FIXATION (if file ends during a fixation) ---
    if current_fix is not None:
        end_time = df.iloc[-1]['time']

        fixations.append({
            'start_time': current_fix['start_time'],
            'end_time': end_time,
            'duration': (end_time - current_fix['start_time']).total_seconds(),
            'x_mean': np.mean(current_fix['x']),
            'y_mean': np.mean(current_fix['y'])
        })
    
    fixations = pd.DataFrame(fixations)

    if len(fixations) == 0:
        return None

    # --- POST FILTER ---
    # convert from seconds to milliseconds
    fixations['duration_ms'] = fixations['duration'] * 1000
    # remove short fixations
    fixations = fixations[fixations['duration_ms'] >= 100]
    # remove outliers
    fixations = fixations[fixations['duration_ms'] < 4000]

    fixations["mid_time"] = fixations["start_time"] + (
        fixations["end_time"] - fixations["start_time"]
    ) / 2

    return fixations

## Processing All Participants: Gaze → Fixations

In [4]:
gaze_root = "/mnt/raid/emotional_data_raquel/supp_mine/gaze"
output_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations"

for participant in os.listdir(gaze_root):
    participant_path = os.path.join(gaze_root, participant)
    for session in os.listdir(participant_path):
        session_path = os.path.join(participant_path, session)
        gaze_csv = os.path.join(session_path, "gaze.csv")
        if not os.path.exists(gaze_csv):
            continue
        print(f"Processing {participant} / {session}")

        try:
            fixations = process_gaze_to_fixations(gaze_csv)
            if fixations is None:
                print("No fixations found")
                continue

            # --- SAVE PATH ---
            save_dir = os.path.join(output_root, participant)
            os.makedirs(save_dir, exist_ok=True)
            save_path = os.path.join(save_dir, f"{session}_fixations.csv")
            fixations.to_csv(save_path, index=False)
            print(f"Saved → {save_path}")

        except Exception as e:
            print(f"Error in {participant}/{session}: {e}")

Processing sub-OE020 / ses-Norrebro
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE020/ses-Norrebro_fixations.csv
Processing sub-OE020 / ses-Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE020/ses-Nordhavn_fixations.csv
Processing sub-OE005 / ses-Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE005/ses-Nordhavn_fixations.csv
Processing sub-OE005 / ses-Hellerup
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE005/ses-Hellerup_fixations.csv
Processing sub-OE018 / ses-Hellerup
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE018/ses-Hellerup_fixations.csv
Processing sub-OE012 / ses-Norreport
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE012/ses-Norreport_fixations.csv
Processing sub-OE021 / ses-Norrebro
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE021/ses

## Temporal Alignment of Fixations with Video Frames

Fixation (event in time) </p>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓</p>
Find closest moment in video</p>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓</p>
Attach frame + gaze coordinates

In [5]:
# align each fixation to the nearest video frame and gaze sample based on timestamp
def align_fixations(session_path, fixations):

    # load full multimodal dataset > gives access to all streams (gaze, video, etc.)
    datapicker = create_datapicker(path=session_path, schema=build_schema)
    dataset = load_dataset(datapicker.selected_path, schema=build_schema)

    # get raw gaze data
    gaze = dataset.streams.PupilLabs.PupilGaze.data

    # handle duplicate timestamps
    gaze = (
        gaze.groupby(level=0)[["GazeX", "GazeY"]]
        .mean()
        .rename(columns={"GazeX": "gaze_x", "GazeY": "gaze_y"})
    )

    # get valid video frames
    decoded_frames = dataset.streams.PupilLabs.DecodedFrames.data
    decoded_frames = decoded_frames[decoded_frames.Value > 0]

    # for each video frame timestamp → find the closest gaze sample
    gaze = gaze.reindex(decoded_frames.index, method="nearest")

    # add frame numbers
    gaze["frame"] = np.arange(len(gaze))

    # match each fixation to the closest gaze/frame in time
    aligned = pd.merge_asof(
        fixations.sort_values("mid_time"),
        gaze,
        left_on="mid_time",
        right_index=True,
        direction="nearest"
    )

    return aligned

Frames:     8    9    10    11    12 </p>
Fixation:        [-----------]</p>
Mid time: &nbsp;&nbsp;&nbsp;&nbsp;  ↑</p>

You pick → Frame 10

___________

In [6]:
# TODO strealine models implementations
def run_pipeline(process_fn, model_name):

    for participant in os.listdir(data_root):

        participant_path = os.path.join(data_root, participant)
        participant_label = f"sub-{participant}"

        for session in os.listdir(participant_path):

            session_path = os.path.join(participant_path, session)
            session_name = session.split("_")[1]

            fix_path = os.path.join(fix_root, participant_label, f"ses-{session_name}_fixations.csv")

            if not os.path.exists(fix_path):
                continue

            fixations = pd.read_csv(fix_path, parse_dates=["start_time", "end_time", "mid_time"])

            try:
                aligned = align_fixations(session_path, fixations)
            except Exception as e:
                continue

            video_path = os.path.join(session_path, "pupil_video.avi")
            if not os.path.exists(video_path):
                continue

            cap = cv.VideoCapture(video_path)

            results = []

            for _, row in aligned.iterrows():

                frame_id = int(row["frame"])
                x = int(row["gaze_x"])
                y = int(row["gaze_y"])

                cap.set(cv.CAP_PROP_POS_FRAMES, frame_id)
                ret, frame = cap.read()

                if not ret or frame.mean() < 5:
                    continue

                # apply model
                label = process_fn(frame, x, y)

                if label is None:
                    continue

                results.append({
                    "participant": participant,
                    "session": session_name,
                    "frame": frame_id,
                    "x": x,
                    "y": y,
                    "label": label,
                    "model": model_name
                })

            cap.release()

_______

# Baseline 1: CLIP Model Initialization

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32",use_safetensors=True).to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


## Urban Semantic Label Set (COCO + Extensions)

Since CLIP doesn’t detect objects, it chooses the best match among the labels you give it, it needs to have labels already defined.

First I started only with the COCO labels, but then quickly realized that the crops were getting very wrong labels so I decided to add road-like labels as per the below </p>

extra_classes = ["floor", "ground", "pavement", "road", "sidewalk"] </p>

Then, labelling got slightly better (from what I could see in the 5 debug crops per participant), however I felt that the COCO labels were missing a lot of key urban gaze like labels, as so I added more like the below (in addition to what I already previously added):

extra_classes = ["road", "sidewalk", "pavement", "ground", "floor",</p>
    "building", "wall", "tree", "grass", "sky", "pole", "trash can", "shop"]</p>

like this the labeling reached a good enough point for me. Knowing that I didnt want to touch on the already defined COCO labels, since that would be a lot of my own bias.

In [8]:
# ===== LOAD COCO LABELS =====
ann_file = "/home/s243636/master-thesis/notebooks/instances_val2017.json"

coco = COCO(ann_file)
cats = coco.loadCats(coco.getCatIds())

# base classes
class_names = [c["name"] for c in cats]
print(class_names)

# add extra urban/environment classes not present in COCO
extra_classes = [
    "road", "sidewalk", "pavement", "trash can", "shop"
    "building", "wall", "tree", "grass", "sky", "pole", 
]
class_names += extra_classes

# convert class names into natural language prompts for CLIP
labels = [f"a photo of {name}" for name in class_names]
print(len(class_names))

loading annotations into memory...
Done (t=0.52s)
creating index...
index created!
['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
90


## CLIP Inference on Fixation-Centered Image Crops

Should I add a confidence threshold here? Bc right now, whatever how low or not the probabilities are, there will always be a final label returned in CLIP.

In [9]:
# ===== CLIP FUNCTION =====
def classify_crop(crop):
    # fix cropped image colors so CLIP predictions become garbage
    image = Image.fromarray(cv.cvtColor(crop, cv.COLOR_BGR2RGB))

    # prepare image and text inputs for CLIP (compare image against all labels)
    inputs = processor(
        text=labels,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    # running the model: compute similarity scores between image and each text label
    outputs = model(**inputs)
    # convert to probabilities
    probs = outputs.logits_per_image.softmax(dim=1)

    # return the label with highest similarity score
    return class_names[probs.argmax().item()]

    # probs = [0.01, 0.05, 0.90, 0.04]
    # argmax → 2
    # class_names[2] → "tree"

## Frame-wise Fixation Classification Using CLIP

In [10]:
def process_video_with_clip(video_path, aligned):

    cap = cv.VideoCapture(video_path)
    results = []

    # group fixations by frame to avoid reading the same frame multiple times - necessary?
    grouped = aligned.groupby("frame")

    # for each frame, get frame number and all fixations in that frame
    for frame_id, gaze_rows in grouped:

        # moves video pointer to frame id and read it
        cap.set(cv.CAP_PROP_POS_FRAMES, int(frame_id))
        ret, frame = cap.read()
        # if not invalid frame index or corrupted video
        if not ret:
            continue

        # to not account for full black image fixations
        if frame.mean() < 5:
            continue

        # looping over fixations in that frame
        for _, gaze in gaze_rows.iterrows():
            # get fixation coordinates
            x, y = int(gaze["gaze_x"]), int(gaze["gaze_y"])

            # --- crop (locality) ---
            patch_size = 120
            h, w = frame.shape[:2]
            x = np.clip(x, 0, w - 1)
            y = np.clip(y, 0, h - 1)
            x1 = max(0, x - patch_size)
            y1 = max(0, y - patch_size)
            x2 = min(w, x + patch_size)
            y2 = min(h, y + patch_size)
            crop = frame[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            label = classify_crop(crop) # crop → label

            results.append({
                "frame": frame_id,
                "x": x,
                "y": y,
                "label": label
            })

    cap.release()
    return results

## Baseline Implementation 1: CLIP-Based Fixation Classification

for each participant </p>
&nbsp;    for each session</p>
&nbsp;&nbsp;&nbsp;&nbsp;        load fixations</p>
&nbsp;&nbsp;&nbsp;&nbsp;        align with video</p>
&nbsp;&nbsp;&nbsp;&nbsp;        run CLIP on fixation crops</p>
&nbsp;&nbsp;&nbsp;&nbsp;        save results</p>

In [11]:
output_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/clip"
os.makedirs(output_root, exist_ok=True)

for participant in os.listdir(data_root):

    participant_path = os.path.join(data_root, participant)
    participant_label = f"sub-{participant}"

    for session in os.listdir(participant_path):

        session_path = os.path.join(participant_path, session)

        # --- LOAD FIXATIONS ---
        session_name = session.split("_")[1]
        fix_path = os.path.join(fix_root, participant_label, f"ses-{session_name}_fixations.csv")

        if not os.path.exists(fix_path):
            continue

        fixations = pd.read_csv(fix_path, parse_dates=["start_time", "end_time", "mid_time"])

        # --- ALIGN ---
        try:
            aligned = align_fixations(session_path, fixations)
        except Exception as e:
            print(f"Alignment error {participant_label}/{session_name}: {e}")
            continue

        # --- VIDEO ---
        video_path = os.path.join(session_path, "pupil_video.avi")

        if not os.path.exists(video_path):
            continue

        print(f"Processing {participant_label}/{session_name}")

        # --- OUTPUT DIR ---
        save_dir = os.path.join(output_root, participant_label)
        os.makedirs(save_dir, exist_ok=True)

        # =========================
        # DEBUG CROPS (first 5)
        # =========================
        cap = cv.VideoCapture(video_path)

        # loop over first 5 fixations
        for i, (_, row) in enumerate(aligned.head(5).iterrows()):
            try:
                frame_id = int(row["frame"])
                cx = int(row["x_mean"])
                cy = int(row["y_mean"])

                cap.set(cv.CAP_PROP_POS_FRAMES, frame_id)
                ret, frame = cap.read()

                if not ret:
                    continue

                # to not account for full black image fixations
                if frame.mean() < 5:
                    continue
                
                # --- crop (locality) --- if gaze very close to the borders then still centered around it
                patch_size = 120
                h, w = frame.shape[:2]
                cx = np.clip(cx, 0, w - 1)
                cy = np.clip(cy, 0, h - 1)
                x1 = max(0, cx - patch_size)
                y1 = max(0, cy - patch_size)
                x2 = min(w, cx + patch_size)
                y2 = min(h, cy + patch_size)
                crop = frame[y1:y2, x1:x2]

                if crop.size == 0:
                    continue

                label = classify_crop(crop)  # CLIP function

                plt.figure()
                plt.imshow(cv.cvtColor(crop, cv.COLOR_BGR2RGB))
                plt.title(label)
                plt.axis("off")

                plt.savefig(os.path.join(save_dir, f"debug_{session_name}_{i}.png"), bbox_inches="tight")
                plt.close()

            except Exception as e:
                print(f"Debug error: {e}")

        cap.release()

        # =========================
        # FULL CLIP PROCESSING
        # =========================
        try:
            results = process_video_with_clip(video_path, aligned)
        except Exception as e:
            print(f"CLIP error {participant_label}/{session}: {e}")
            continue

        # --- SAVE CSV ---
        save_path = os.path.join(save_dir, f"ses-{session_name}_clip.csv")

        pd.DataFrame(results).to_csv(save_path, index=False)

        print(f"Saved → {save_path}")

Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE011/Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/clip/sub-OE011/ses-Nordhavn_clip.csv
Processing sub-OE015/Norreport
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/clip/sub-OE015/ses-Norreport_clip.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE019/Hellerup
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/clip/sub-OE019/ses-Hellerup_clip.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE010/Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/clip/sub-OE010/ses-Nordhavn_clip.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE024/Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/clip/sub-

# Baseline 2: CLIP (Gaussian) Model Initialization

Fixation</p>
&nbsp;&nbsp;&nbsp;↓</p>
Local crop (removes noise)</p>
&nbsp;&nbsp;&nbsp;↓</p>
Gaussian (soft focus)</p>
&nbsp;&nbsp;&nbsp;↓</p>
CLIP (semantic label)

In [12]:
def gaussian_focus(frame, cx, cy, sigma=80): # 80 gives a good balanced effect
    # get image size
    h, w = frame.shape[:2]

    # create coordinate grid
    x = np.arange(w)
    y = np.arange(h)
    xx, yy = np.meshgrid(x, y)

    # gaussian mask > distance from fixation → convert to weight
    mask = np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sigma**2))
    mask = mask[..., None]  # make 3-channel

    # blur full image
    blurred = cv.GaussianBlur(frame, (51, 51), 0)

    # combine sharp + blurred
    focused = frame * mask + blurred * (1 - mask)
    # center → clear ; edges → blurry 

    return focused.astype(np.uint8)

Fixation → crop (local region) → apply Gaussian inside crop → CLIP → label

Switched from applying Gaussian to the full frame to using it within a cropped region around the fixation. The full-frame approach gave too much weight to the overall scene (e.g., pavement, instead of understanding that the user was actually focusing on the bikes within the pavement), so adding a crop helps focus on the actual gaze target while still keeping some surrounding context through the Gaussian.

In [13]:
def process_video_with_clip_gaussian_crop(video_path, aligned):

    cap = cv.VideoCapture(video_path)
    results = []

    grouped = aligned.groupby("frame")

    for frame_id, gaze_rows in grouped:

        cap.set(cv.CAP_PROP_POS_FRAMES, int(frame_id))
        ret, frame = cap.read()
        if not ret:
            continue

        # to not account for full black image fixations
        if frame.mean() < 5:
            continue

        for _, gaze in gaze_rows.iterrows():
            # get fixation coordinates
            x, y = int(gaze["gaze_x"]), int(gaze["gaze_y"])

            # --- crop (locality) --- 
            patch_size = 200
            h, w = frame.shape[:2]
            x = np.clip(x, 0, w - 1)
            y = np.clip(y, 0, h - 1)
            x1 = max(0, x - patch_size)
            y1 = max(0, y - patch_size)
            x2 = min(w, x + patch_size)
            y2 = min(h, y + patch_size)
            crop = frame[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            # --- gaussian INSIDE crop ---
            # center of crop = fixation point
            cx_crop = x - x1
            cy_crop = y - y1
            focused = gaussian_focus(crop, cx_crop, cy_crop)

            # --- classify ---
            label = classify_crop(focused)

            results.append({
                "frame": frame_id,
                "x": x,
                "y": y,
                "label": label
            })

    cap.release()
    return results

## Baseline Implementation 2: Gaussian CLIP-Based Fixation Classification

In [14]:
output_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/clip_gaussian"

os.makedirs(output_root, exist_ok=True)

for participant in os.listdir(data_root):

    participant_path = os.path.join(data_root, participant)
    participant_label = f"sub-{participant}"

    for session in os.listdir(participant_path):

        session_path = os.path.join(participant_path, session)

        # --- LOAD FIXATIONS ---
        session_name = session.split("_")[1]
        fix_path = os.path.join(fix_root, participant_label, f"ses-{session_name}_fixations.csv")

        if not os.path.exists(fix_path):
            continue

        fixations = pd.read_csv(fix_path, parse_dates=["start_time", "end_time", "mid_time"])

        # --- ALIGN fixations to frames ---
        try:
            aligned = align_fixations(session_path, fixations)
        except Exception as e:
            print(f"Alignment error {participant_label}/{session_name}: {e}")
            continue

        # --- VIDEO ---
        video_path = os.path.join(session_path, "pupil_video.avi")

        if not os.path.exists(video_path):
            continue

        print(f"Processing {participant_label}/{session_name}")

        # --- OUTPUT DIR ---
        save_dir = os.path.join(output_root, participant_label)
        os.makedirs(save_dir, exist_ok=True)

        # =========================
        # DEBUG CROPS (first 5)
        # =========================
        cap = cv.VideoCapture(video_path)
        patch_size = 200

        for i, (_, row) in enumerate(aligned.head(5).iterrows()):
            try:
                frame_id = int(row["frame"])
                cx = int(row["x_mean"])
                cy = int(row["y_mean"])

                cap.set(cv.CAP_PROP_POS_FRAMES, frame_id)
                ret, frame = cap.read()

                if not ret:
                    continue

                # to not account for full black image fixations
                if frame.mean() < 5:
                    continue

                # define crop properly - if gaze very close to the borders then still centered around it
                h, w = frame.shape[:2]
                cx = np.clip(cx, 0, w - 1)
                cy = np.clip(cy, 0, h - 1)
                x1 = max(0, cx - patch_size)
                y1 = max(0, cy - patch_size)
                x2 = min(w, cx + patch_size)
                y2 = min(h, cy + patch_size)
                crop = frame[y1:y2, x1:x2]

                if crop.size == 0:
                    continue

                # apply gaussian on crop
                cx_crop = cx - x1
                cy_crop = cy - y1
                focused = gaussian_focus(crop, cx_crop, cy_crop)
                label = classify_crop(focused)

                plt.figure()
                plt.imshow(cv.cvtColor(focused, cv.COLOR_BGR2RGB))
                plt.title(label)
                plt.axis("off")

                plt.savefig(os.path.join(save_dir, f"debug_{session_name}_{i}.png"), bbox_inches="tight")
                plt.close()

            except Exception as e:
                print(f"Debug error: {e}")

        cap.release()

        # =========================
        # FULL CLIP PROCESSING
        # =========================
        try:
            results = process_video_with_clip_gaussian_crop(video_path, aligned)
        except Exception as e:
            print(f"CLIP error {participant_label}/{session}: {e}")
            continue

        # --- SAVE CSV ---
        save_path = os.path.join(save_dir, f"ses-{session_name}_clip_gaussian.csv")

        pd.DataFrame(results).to_csv(save_path, index=False)

        print(f"Saved → {save_path}")

Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE011/Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/clip_gaussian/sub-OE011/ses-Nordhavn_clip_gaussian.csv
Processing sub-OE015/Norreport
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/clip_gaussian/sub-OE015/ses-Norreport_clip_gaussian.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE019/Hellerup
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/clip_gaussian/sub-OE019/ses-Hellerup_clip_gaussian.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE010/Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/clip_gaussian/sub-OE010/ses-Nordhavn_clip_gaussian.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE024/Nordhavn
Saved → /mnt

_____________

# Main Model: Mask2Former Model Initialization

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "facebook/mask2former-swin-large-cityscapes-semantic"
processor = Mask2FormerImageProcessor.from_pretrained(model_id)
model = Mask2FormerForUniversalSegmentation.from_pretrained(model_id).to(device)

In [16]:
output_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/segmentation"
os.makedirs(output_root, exist_ok=True)
id2label = model.config.id2label

for participant in os.listdir(data_root):

    participant_path = os.path.join(data_root, participant)
    participant_label = f"sub-{participant}"

    for session in os.listdir(participant_path):

        session_path = os.path.join(participant_path, session)

        # --- LOAD FIXATIONS ---
        session_name = session.split("_")[1]
        fix_path = os.path.join(fix_root, participant_label, f"ses-{session_name}_fixations.csv")

        if not os.path.exists(fix_path):
            continue

        fixations = pd.read_csv(fix_path, parse_dates=["start_time", "end_time", "mid_time"])

        # --- ALIGN ---
        try:
            aligned = align_fixations(session_path, fixations)
        except Exception as e:
            print(f"Alignment error {participant_label}/{session_name}: {e}")
            continue

        # --- VIDEO ---
        video_path = os.path.join(session_path, "pupil_video.avi")

        if not os.path.exists(video_path):
            continue

        print(f"Processing {participant_label}/{session_name}")

        # --- OUTPUT DIR ---
        save_dir = os.path.join(output_root, participant_label)
        os.makedirs(save_dir, exist_ok=True)

        cap = cv.VideoCapture(video_path)
        grouped = aligned.groupby("frame")

        results = []

        # =========================
        # PROCESS ALL FRAMES
        # =========================
        for i, (frame_id, gaze_rows) in enumerate(grouped):

            cap.set(cv.CAP_PROP_POS_FRAMES, int(frame_id))
            ret, frame = cap.read()

            if not ret:
                continue

            # skip black frames
            if frame.mean() < 5:
                continue

            frame_rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)

            inputs = processor(images=frame_rgb, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)

            segmentation = processor.post_process_semantic_segmentation(
                outputs,
                target_sizes=[frame_rgb.shape[:2]]
            )[0]

            for j, (_, fix) in enumerate(gaze_rows.iterrows()):

                if pd.isna(fix["gaze_x"]) or pd.isna(fix["gaze_y"]):
                    continue

                x = int(fix["x_mean"])
                y = int(fix["y_mean"])

                h, w = segmentation.shape
                x = np.clip(x, 0, w - 1)
                y = np.clip(y, 0, h - 1)

                label = segmentation[y, x].item()
                label_name = id2label[label]

                # --- SAVE RESULT ---
                results.append({
                    "participant": participant,
                    "session": session_name,
                    "frame": frame_id,
                    "x": x,
                    "y": y,
                    "label_id": label,
                    "label_name": label_name
                })

                # =========================
                # DEBUG (first 5 fixations only)
                # =========================
                if len(results) <= 5:
                    frame_vis = frame.copy()
                    cv.circle(frame_vis, (x, y), 6, (0, 0, 255), -1)

                    plt.figure(figsize=(5,5))
                    plt.imshow(cv.cvtColor(frame_vis, cv.COLOR_BGR2RGB))
                    plt.title(f"{session_name} → {label_name}")
                    plt.axis("off")

                    plt.savefig(
                        os.path.join(save_dir, f"debug_{session_name}_{len(results)}.png"),
                        bbox_inches="tight"
                    )
                    plt.close()

        cap.release()

        # =========================
        # SAVE CSV
        # =========================
        if len(results) > 0:
            save_path = os.path.join(save_dir, f"ses-{session_name}_segmentation.csv")
            pd.DataFrame(results).to_csv(save_path, index=False)
            print(f"Saved → {save_path}")

Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE011/Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/segmentation/sub-OE011/ses-Nordhavn_segmentation.csv
Processing sub-OE015/Norreport
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/segmentation/sub-OE015/ses-Norreport_segmentation.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE019/Hellerup
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/segmentation/sub-OE019/ses-Hellerup_segmentation.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE010/Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/segmentation/sub-OE010/ses-Nordhavn_segmentation.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE024/Nordhavn
Saved → /mnt/raid/em

___________

## Participant video with labeling for understanding

In [2]:
# -----------------------------
# PATHS
# -----------------------------
video_path = "/mnt/raid/emotional_data_raquel/data/OE012/Copenhagen_Norreport_sub-OE202012_2024-06-26T125140Z/pupil_video.avi"
seg_path = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/segmentation/sub-OE012/ses-Norreport_segmentation.csv"
output_path = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/segmentation/sub-OE012/gaze_overlay.mp4"

# -----------------------------
# LOAD DATA
# -----------------------------
seg = pd.read_csv(seg_path)

seg = seg[
    (seg["participant"] == "OE012") &
    (seg["session"] == "Norreport")
]

# -----------------------------
# VIDEO SETUP
# -----------------------------
cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

frame_idx = 0

# -----------------------------
# COLOR MAP
# -----------------------------
colors = {
    "road": (128, 64, 128),
    "person": (0, 0, 255),
    "bicycle": (255, 0, 0),
    "car": (0, 255, 255),
}

# -----------------------------
# LOOP
# -----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    rows = seg[seg["frame"] == frame_idx]

    for _, row in rows.iterrows():
        x = int(row["x"])
        y = int(row["y"])
        label = row["label_name"]

        color = colors.get(label, (255, 255, 255))

        # safety clipping
        x = max(0, min(x, w - 1))
        y = max(0, min(y, h - 1))

        cv2.circle(frame, (x, y), 6, color, -1)

        cv2.putText(
            frame,
            label,
            (x + 10, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255,255,255),
            2
        )

    out.write(frame)

    frame_idx += 1

cap.release()
out.release()

print(f"Saved video to: {output_path}")

Saved video to: gaze_overlay.mp4


In [1]:
import cv2
import pandas as pd

# -----------------------------
# PATHS
# -----------------------------
video_path = "/mnt/raid/emotional_data_raquel/data/OE012/Copenhagen_Norreport_sub-OE202012_2024-06-26T125140Z/pupil_video.avi"
seg_path = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/segmentation/sub-OE012/ses-Norreport_segmentation.csv"
output_path = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/segmentation/sub-OE012/gaze_overlay_slow.mp4"

# -----------------------------
# SETTINGS
# -----------------------------
SECONDS = 10
SLOW_FACTOR = 4   # 4 = 4x slower

# -----------------------------
# LOAD DATA
# -----------------------------
seg = pd.read_csv(seg_path)

seg = seg[
    (seg["participant"] == "OE012") &
    (seg["session"] == "Norreport")
]

# -----------------------------
# VIDEO SETUP
# -----------------------------
cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

max_frames = int(fps * SECONDS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

frame_idx = 0

# -----------------------------
# COLOR MAP
# -----------------------------
colors = {
    "road": (128, 64, 128),
    "person": (0, 0, 255),
    "bicycle": (255, 0, 0),
    "car": (0, 255, 255),
}

# -----------------------------
# LOOP
# -----------------------------
while True:
    ret, frame = cap.read()
    if not ret or frame_idx >= max_frames:
        break

    rows = seg[seg["frame"] == frame_idx]

    for _, row in rows.iterrows():
        x = int(row["x"])
        y = int(row["y"])
        label = row["label_name"]

        color = colors.get(label, (255, 255, 255))

        x = max(0, min(x, w - 1))
        y = max(0, min(y, h - 1))

        cv2.circle(frame, (x, y), 6, color, -1)

        cv2.putText(
            frame,
            label,
            (x + 10, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255,255,255),
            2
        )

    # -----------------------------
    # SLOW DOWN HERE
    # -----------------------------
    for _ in range(SLOW_FACTOR):
        out.write(frame)

    frame_idx += 1

cap.release()
out.release()

print(f"Saved video to: {output_path}")

Saved video to: /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/segmentation/sub-OE012/gaze_overlay_slow.mp4
